# Level 3 · Task 1 — Random Forest Classifier

**Goal:** predict telecom churn with a random forest — an ensemble of many
decision trees — and see if it beats the single models we built before.

We'll follow through on the same churn problem from Level 2 Task 1:
1. load + preprocess (same cleaning; **no scaling needed for trees**)
2. tune hyperparameters — `n_estimators`, `max_depth`
3. cross-validation + full test evaluation (precision / recall / F1)
4. feature importance analysis

*Tools: Python, scikit-learn, pandas, matplotlib*

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option('display.width', 160)
pd.set_option('display.max_columns', 30)

train = pd.read_csv('../data/churn-bigml-80.csv')
test = pd.read_csv('../data/churn-bigml-20.csv')

def preprocess(df):
    '''identical cleaning to the logistic regression notebook'''
    df = df.copy()
    y = df['Churn'].astype(str).str.strip().map({'True': 1, 'False': 0})
    df['International plan'] = (df['International plan'].str.strip() == 'Yes').astype(int)
    df['Voice mail plan'] = (df['Voice mail plan'].str.strip() == 'Yes').astype(int)
    df = df.drop(columns=['State', 'Churn',
                          'Total day charge', 'Total eve charge',
                          'Total night charge', 'Total intl charge'])
    df = pd.get_dummies(df, columns=['Area code'], prefix='area', dtype=int)
    return df, y

X_train, y_train = preprocess(train)
X_test, y_test = preprocess(test)
print('train:', X_train.shape, ' test:', X_test.shape)
print('churn rate: {:.1f}%'.format(100 * y_train.mean()))
X_train.head()

train: (2666, 16)  test: (667, 16)
churn rate: 14.6%


,Account length,International plan,Voice mail plan,Number vmail messages,Total day minutes,Total day calls,Total eve minutes,Total eve calls,Total night minutes,Total night calls,Total intl minutes,Total intl calls,Customer service calls,area_408,area_415,area_510
0,128,0,1,25,265.1,110,197.4,99,244.7,91,10.0,3,1,0,1,0
1,107,0,1,26,161.6,123,195.5,103,254.4,103,13.7,3,1,0,1,0
2,137,0,0,0,243.4,114,121.2,110,162.6,104,12.2,5,0,0,1,0
3,84,1,0,0,299.4,71,61.9,88,196.9,89,6.6,7,2,1,0,0
4,75,1,0,0,166.7,113,148.3,122,186.9,121,10.1,3,3,0,1,0


In [2]:
# baseline #1: a single decision tree (our Level 2 Task 2 model)
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import f1_score

tree = DecisionTreeClassifier(random_state=42).fit(X_train, y_train)
tree_test_f1 = f1_score(y_test, tree.predict(X_test))

print(f'single tree   -> test accuracy {tree.score(X_test, y_test):.3f}, churn F1 {tree_test_f1:.3f}')
print(f'(for reference, logistic regression from Level 2 Task 1 scored 0.853 accuracy, 0.26 F1)')

single tree   -> test accuracy 0.909, churn F1 0.690
(for reference, logistic regression from Level 2 Task 1 scored 0.853 accuracy, 0.26 F1)
